# ⚽ FIFA World Cup 2026 — Live Dashboard

A **Spur App** that fuses two live data sources into one reactive dashboard:

- **Polymarket** (`gamma-api.polymarket.com`) — prediction-market *implied odds* for World Cup 2026 questions (winner, hosts, golden boot).
- **RSSHub** (`rsshub.app`, Google-News fallback) — the World Cup headline feed.

**Pipeline (reactive DAG):** two Python *source* cells pull each feed and publish an Arrow **port** (`wc_markets`, `wc_news`); the **Deno frontend cell** reads those ports and renders a **Perspective** datagrid + chart, with the live MCP tool `wc_snapshot` preferred when the plugin is running.

> Re-run a source cell (or arm a cron schedule on it) → the cascade re-renders the dashboard. In **App mode**, only the frontend cell is shown. Implied probabilities are *market prices*, not forecasts.

In [17]:
import os, sys

# Reuse the app's own data layer (server/worldcup.py) — the exact code the
# wc_markets / wc_snapshot MCP tools call. Single source of truth.
for _cand in (os.path.join(os.getcwd(), "server"),
              os.path.join(os.path.dirname(os.getcwd()), "server")):
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand)
        break
import worldcup

# --- Polymarket data source -------------------------------------------------
markets = worldcup.fetch_markets("world cup", 50)
live = bool(markets)
if not markets:
    # Offline / no active market yet — labelled sample so the dashboard renders.
    _sample = [
        ("[sample] Will Spain win the 2026 World Cup?",     "0.18", "4200000"),
        ("[sample] Will Argentina win the 2026 World Cup?", "0.15", "3900000"),
        ("[sample] Will France win the 2026 World Cup?",    "0.14", "3500000"),
        ("[sample] Will Brazil win the 2026 World Cup?",    "0.13", "3300000"),
        ("[sample] Will England win the 2026 World Cup?",   "0.11", "2600000"),
    ]
    markets = [
        worldcup.shape_market(
            {"question": q, "outcomes": '["Yes","No"]',
             "outcomePrices": f'["{p}","{round(1 - float(p), 2)}"]',
             "volume": v, "slug": ""}
        )
        for q, p, v in _sample
    ]

# Prove the Polymarket spur_rest DuckDB datasource is wired (cheap LIMIT 1 probe
# — never count(*), which would force the table function to paginate the whole
# active-market set).
try:
    import duckdb
    _ext = os.path.expanduser("~/.spur/extensions/spur_rest.duckdb_extension")
    _con = duckdb.connect(config={"allow_unsigned_extensions": "true"})
    _con.execute(f"LOAD '{_ext}'")
    _con.execute("SELECT question FROM polymarket_markets() LIMIT 1").fetchall()
    print("polymarket datasource OK: spur_rest table function reachable")
    _con.close()
except Exception as _e:  # noqa: BLE001
    print(f"(polymarket datasource probe skipped: {_e})")

spur.put("wc_markets", markets)
print(f"{len(markets)} World Cup markets ({'LIVE' if live else 'SAMPLE'}) -> port wc_markets")
markets[:3]

polymarket datasource OK: spur_rest table function reachable


question,outcome,yes_prob,implied_pct,volume,volume_24hr,liquidity,end_date,url
Will Uruguay win Group H in the 2026 FIFA World Cup?,Yes,0.205,20.5,99343.75193399987,1455.678451,53407.5126,2026-06-27,https://polymarket.com/market/will-uruguay-win-group-h-in-the-2026-fifa-world-cup
Will a player representing Turkiye be the top goalscorer at the 2026 FIFA World Cup?,Yes,0.0025,0.2,9901.563873999998,987.49,8649.0973,2026-08-20,https://polymarket.com/market/will-a-player-representing-turkiye-be-the-top-goalscorer-at-the-2026-fifa-world-cup
Will Paraguay win the Fair Play Award for the 2026 FIFA World Cup?,Yes,0.005,0.5,999.6916659999999,57.0,7217.42112,2026-07-20,https://polymarket.com/market/will-paraguay-win-the-fair-play-award-for-the-2026-fifa-world-cup-20260603201521221
Will Netherlands finish second in Group F in the 2026 FIFA World Cup Group Stage?,Yes,0.335,33.5,999.0935439999998,9.0,276.3035,2026-07-12,https://polymarket.com/market/will-netherlands-finish-second-in-group-f-in-the-2026-fifa-world-cup-group-stage-20260605151901145
Will 4+ matches go to extra time during the 2026 FIFA World Cup?,Yes,0.951,95.1,998.8122479999997,21.355422,2780.73741,2026-07-20,https://polymarket.com/market/will-4-matches-go-to-extra-time-during-the-2026-fifa-world-cup-20260610205949023


11 World Cup markets (LIVE) -> port wc_markets


[{'question': 'Will Uruguay win Group H in the 2026 FIFA World Cup?',
  'outcome': 'Yes',
  'yes_prob': 0.205,
  'implied_pct': 20.5,
  'volume': 99343.75193399987,
  'volume_24hr': 1455.678451,
  'liquidity': 53407.5126,
  'end_date': '2026-06-27',
  'url': 'https://polymarket.com/market/will-uruguay-win-group-h-in-the-2026-fifa-world-cup'},
 {'question': 'Will a player representing Turkiye be the top goalscorer at the 2026 FIFA World Cup?',
  'outcome': 'Yes',
  'yes_prob': 0.0025,
  'implied_pct': 0.2,
  'volume': 9901.563873999998,
  'volume_24hr': 987.49,
  'liquidity': 8649.0973,
  'end_date': '2026-08-20',
  'url': 'https://polymarket.com/market/will-a-player-representing-turkiye-be-the-top-goalscorer-at-the-2026-fifa-world-cup'},
 {'question': 'Will Paraguay win the Fair Play Award for the 2026 FIFA World Cup?',
  'outcome': 'Yes',
  'yes_prob': 0.005,
  'implied_pct': 0.5,
  'volume': 999.6916659999999,
  'volume_24hr': 57.0,
  'liquidity': 7217.42112,
  'end_date': '2026-07-2

In [16]:
import os, sys

for _cand in (os.path.join(os.getcwd(), "server"),
              os.path.join(os.path.dirname(os.getcwd()), "server")):
    if os.path.isdir(_cand):
        sys.path.insert(0, _cand)
        break
import worldcup

# --- RSSHub data source (rsshub.app first, Google-News RSS fallback) --------
news = worldcup.fetch_news(30)
live_news = bool(news)
if not news:
    news = [
        {"title": "[sample] World Cup 2026 host cities finalize match schedule",
         "link": "https://example.org/wc/1", "published": "", "source": "sample"},
        {"title": "[sample] Qualification race tightens across confederations",
         "link": "https://example.org/wc/2", "published": "", "source": "sample"},
        {"title": "[sample] Ticket demand sets a record for the opening match",
         "link": "https://example.org/wc/3", "published": "", "source": "sample"},
    ]

spur.put("wc_news", news)
_src = news[0]["source"] if news else "none"
print(f"{len(news)} headlines ({'LIVE via ' + _src if live_news else 'SAMPLE'}) -> port wc_news")
news[:3]

title,link,published,source
"World Cup 2026 live updates: Colombia returns against debutant Uzbekistan; Ghana beats Panama on late goal, England downs Croatia, Portugal and DR Congo draw in earlier games - NBC News",https://news.google.com/rss/articles/CBMingFBVV95cUxPVlQ4X25LdktwamxGUFVOSUl6bDVrc0FqVlBaWURvM3c3bXp4X0xMSTd1RmM1WEhDX1E4UmgtdlREVnlyZWU5OEdMdHh2YmtLWTU4Vks0UVEtVXlKMDVLMV81QjhmWFpDRG42NE1DMWlMREwxVWxWeEQwNmxZVVJISWd4empWazJJQXRaMnlTNThpbDJoU0hfWS10dUVLdw?oc=5,"Thu, 18 Jun 2026 02:37:00 GMT",rss
Argentina vs Algeria Extended Highlights | 2026 FIFA World Cup™ - FOX Sports,https://news.google.com/rss/articles/CBMiYkFVX3lxTE02NHFzQldYU2xRODhac1pYYUtFMGJoaUdzNFRQYWNCSHlzQk9oOGlBeTExYlphWjRmYVk3UDdGZXNLRkJYamVpQklRZFFxYnY2eVVtRFRhRk1nZDVVMlNiTWhB?oc=5,"Wed, 17 Jun 2026 03:29:07 GMT",rss
Ghana beat Panama with stoppage-time winner at World Cup 2026: Live updates and reaction - The New York Times,https://news.google.com/rss/articles/CBMitAFBVV95cUxPajdnX1o2N29JSlRqUEtvWWlUcHdIWWJ1SzhVZG90Wl9XcDZRdlVPaWREN3hRSmstdFdfTjl6RV9MVXVkclJmVUwtT3RNSHdocmtMV2hGUlViUzE2MjdXRkhxQkFZVE83NXN2bVBJQlBlaXVfRTZsMTZaZnZNU3l1dUIyZVBrRVhFa2pJcjAxby1Yc3prTE1QbHdSY2ZUTkRyTWd6a2Voa3lRa3lnZDJYNTZRSnY?oc=5,"Thu, 18 Jun 2026 02:32:00 GMT",rss
"FIFA World Cup 2026 MD7 recap: Ronaldo's Portugal frustrated by Congo DR, England hit 4 past Croatia, Panama hold Ghana - ESPN",https://news.google.com/rss/articles/CBMizAFBVV95cUxNLVZmUXdLZ2gtNWF4SlZPWXVqSkRIeGhjWXAyREZkbjU5QnBGNVJoX3ZIb1pUeTFpdEtmX0NvT3d0R0pKTlNKR2JYOEk2eXhFVG4yM19zSGFZSk5EZTRPcnlVT3h6VUxBMTU0ZmJadV9tTjhnNEs3QTdHbXdTNjBYTThzRXAwWXd4VEpSYWtXV09iVndnY0RRcWJac040emtMWk1TeHV4WnZHWXE4Nnl1S0lMTHBtcEhzbmt0dXdmTEVuRkQtWlJJajhpV3I?oc=5,"Thu, 18 Jun 2026 01:09:00 GMT",rss
Uzbekistan v Colombia: World Cup 2026 – live - The Guardian,https://news.google.com/rss/articles/CBMi3wFBVV95cUxPZl9mNXBrcjhVNTBPRmZZcTdXNEtlUjRBR2V1UTVYLVJHMUVKckdXUDVfakpyamhzaU5keWhnR3dNV0dWX3lLOXFCUEMxNF8tM1BnQ296XzBZZWRSbmRaX0VCMnFIdzZucDNLZ0dyWjI4dEUxLWxDSUpuTWh3cWZ3UHhiOWp3cUR4LXBxZDQ0Y3dEb20yZ2tqRVltOV84VGNHSnc4UHBsbk0zXzJaZUtqV0x3SWZvNFRQUEpxaFZLZDN1UWV6QzZfM2RJV2FGUGRZWklRZFVSMFdJYV84bXo0?oc=5,"Thu, 18 Jun 2026 01:30:51 GMT",rss


30 headlines (LIVE via rss) -> port wc_news


[{'title': 'World Cup 2026 live updates: Colombia returns against debutant Uzbekistan; Ghana beats Panama on late goal, England downs Croatia, Portugal and DR Congo draw in earlier games - NBC News',
  'link': 'https://news.google.com/rss/articles/CBMingFBVV95cUxPVlQ4X25LdktwamxGUFVOSUl6bDVrc0FqVlBaWURvM3c3bXp4X0xMSTd1RmM1WEhDX1E4UmgtdlREVnlyZWU5OEdMdHh2YmtLWTU4Vks0UVEtVXlKMDVLMV81QjhmWFpDRG42NE1DMWlMREwxVWxWeEQwNmxZVVJISWd4empWazJJQXRaMnlTNThpbDJoU0hfWS10dUVLdw?oc=5',
  'published': 'Thu, 18 Jun 2026 02:37:00 GMT',
  'source': 'rss'},
 {'title': 'Argentina vs Algeria Extended Highlights | 2026 FIFA World Cup™ - FOX Sports',
  'link': 'https://news.google.com/rss/articles/CBMiYkFVX3lxTE02NHFzQldYU2xRODhac1pYYUtFMGJoaUdzNFRQYWNCSHlzQk9oOGlBeTExYlphWjRmYVk3UDdGZXNLRkJYamVpQklRZFFxYnY2eVVtRFRhRk1nZDVVMlNiTWhB?oc=5',
  'published': 'Wed, 17 Jun 2026 03:29:07 GMT',
  'source': 'rss'},
 {'title': 'Ghana beat Panama with stoppage-time winner at World Cup 2026: Live updates and reaction - The 

In [13]:
// World Cup 2026 — Perspective dashboard (frontend cell).
// Renders purely from the wc_markets / wc_news Arrow ports, which the two
// source cells populate reactively. To refresh, re-run a source cell (or arm a
// cron schedule on it) — the cascade re-renders this cell. We deliberately do
// NOT call an MCP tool here: that would put a blocking network + app-plugin
// round-trip in the App-mode render path (slow on open, can stall if the
// plugin is still spawning). The ports already hold the data.

function rowsOf(t) { try { return t && t.toArray ? t.toArray() : []; } catch (_) { return []; } }
function coerceMarket(r) {
  return {
    question: String(r.question ?? ""),
    implied_pct: r.implied_pct == null ? null : Number(r.implied_pct),
    volume: Number(r.volume ?? 0),
    end_date: String(r.end_date ?? ""),
    url: String(r.url ?? ""),
  };
}
function coerceNews(r) {
  return {
    title: String(r.title ?? ""),
    link: String(r.link ?? ""),
    published: String(r.published ?? ""),
    source: String(r.source ?? ""),
  };
}

const markets = rowsOf(spur.get("wc_markets")).map(coerceMarket);
const news = rowsOf(spur.get("wc_news")).map(coerceNews);

const priced = markets.filter((m) => m.implied_pct != null);
const favorite = priced.slice().sort((a, b) => b.implied_pct - a.implied_pct)[0] || null;
const totalVol = markets.reduce((s, m) => s + (m.volume || 0), 0);
const isSample = markets.some((m) => m.question.startsWith("[sample]")) || news.some((n) => n.source === "sample");

const esc = (s) => String(s == null ? "" : s).replace(/[&<>"]/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;" }[c]));
const fmtVol = (v) => "$" + (Number(v) || 0).toLocaleString("en-US", { maximumFractionDigits: 0 });
const fmtPct = (v) => (v == null ? "—" : Number(v).toFixed(1) + "%");

const kpiCards = [
  ["Market favorite", favorite ? esc(favorite.question.replace(/^\[sample\]\s*/, "")) : "—", favorite ? fmtPct(favorite.implied_pct) : ""],
  ["Markets tracked", String(markets.length), ""],
  ["Total volume", fmtVol(totalVol), ""],
  ["Headlines", String(news.length), ""],
].map(([label, big, sub]) => `<div class="kpi"><div class="kpi-label">${label}</div><div class="kpi-big">${big}</div><div class="kpi-sub">${sub}</div></div>`).join("");

const newsItems = news.slice(0, 18).map((n) => `<li><a href="${esc(n.link)}" target="_blank" rel="noopener">${esc(n.title)}</a><span class="src">${esc(n.source)}</span></li>`).join("") || '<li class="muted">No headlines.</li>';

// Browser-side module script. NOTE: no backticks / no ${} inside INNER so the
// outer template literal does not interpolate it.
const INNER = `
const fmtPct = (v) => (v == null ? "—" : Number(v).toFixed(1) + "%");
const fmtVol = (v) => "$" + (Number(v) || 0).toLocaleString("en-US", { maximumFractionDigits: 0 });
const esc = (s) => String(s == null ? "" : s).replace(/[&<>]/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;" }[c]));
function fallbackTable(rows) {
  let h = "<table class='tbl'><thead><tr><th>Market</th><th class='num'>Implied</th><th class='num'>Volume</th><th>Ends</th></tr></thead><tbody>";
  for (const r of rows) { h += "<tr><td>" + esc(r.question) + "</td><td class='num'>" + fmtPct(r.implied_pct) + "</td><td class='num'>" + fmtVol(r.volume) + "</td><td>" + esc(r.end_date || "") + "</td></tr>"; }
  return h + "</tbody></table>";
}
const container = document.getElementById("pv");
const data = MARKETS.length ? MARKETS : [{ question: "(no markets)", implied_pct: null, volume: 0, end_date: "", url: "" }];
let ok = false;
for (const v of ["3.7.0", "3.3.0"]) {
  try {
    const base = "https://cdn.jsdelivr.net/npm/@finos/";
    const perspective = (await import(base + "perspective@" + v + "/dist/cdn/perspective.js")).default;
    await import(base + "perspective-viewer@" + v + "/dist/cdn/perspective-viewer.js");
    await import(base + "perspective-viewer-datagrid@" + v + "/dist/cdn/perspective-viewer-datagrid.js");
    await import(base + "perspective-viewer-d3fc@" + v + "/dist/cdn/perspective-viewer-d3fc.js");
    const worker = await perspective.worker();
    const table = await worker.table(data);
    const viewer = document.createElement("perspective-viewer");
    container.innerHTML = "";
    container.appendChild(viewer);
    await viewer.load(table);
    await viewer.restore({ plugin: "Datagrid", columns: ["question", "implied_pct", "volume", "end_date"], sort: [["volume", "desc"]], theme: "Pro Dark" });
    ok = true;
    break;
  } catch (e) { /* try next version */ }
}
if (!ok) { container.classList.add("fallback"); container.innerHTML = fallbackTable(data); }
`;

const safeMarkets = JSON.stringify(markets).replace(/</g, "\\u003c");
const scriptTag = '<scr' + 'ipt type="module">\nconst MARKETS = ' + safeMarkets + ';\n' + INNER + '\n</scr' + 'ipt>';

const html = `<!doctype html><html><head><meta charset="utf-8">
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@finos/perspective-viewer@3.7.0/dist/css/themes.css">
<style>
  :root { color-scheme: dark; }
  * { box-sizing: border-box; }
  body { margin: 0; background: #0a0e1a; color: #e6edf3; font-family: Inter, system-ui, sans-serif; }
  .wrap { max-width: 1180px; margin: 0 auto; padding: 22px; }
  .top { display: flex; align-items: baseline; gap: 12px; flex-wrap: wrap; }
  h1 { font-size: 22px; margin: 0; }
  .badge { font-size: 11px; font-weight: 700; padding: 3px 9px; border-radius: 999px; letter-spacing: .04em; }
  .badge.live { background: #0f3d2e; color: #34d399; }
  .badge.sample { background: #3d2f0f; color: #fbbf24; }
  .sub { color: #93a1b3; font-size: 13px; margin: 4px 0 18px; }
  .kpis { display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; margin-bottom: 18px; }
  .kpi { background: #121829; border: 1px solid #1e2740; border-radius: 14px; padding: 14px 16px; }
  .kpi-label { color: #93a1b3; font-size: 12px; text-transform: uppercase; letter-spacing: .05em; }
  .kpi-big { font-size: 19px; font-weight: 700; margin-top: 6px; line-height: 1.25; }
  .kpi-sub { color: #34d399; font-weight: 700; margin-top: 2px; }
  .grid { display: grid; grid-template-columns: 1.7fr 1fr; gap: 16px; align-items: start; }
  .card { background: #121829; border: 1px solid #1e2740; border-radius: 14px; overflow: hidden; }
  .card h2 { font-size: 13px; text-transform: uppercase; letter-spacing: .06em; color: #93a1b3; margin: 0; padding: 14px 16px; border-bottom: 1px solid #1e2740; }
  #pv { height: 460px; width: 100%; }
  #pv.fallback { height: auto; max-height: 460px; overflow: auto; }
  .tbl { width: 100%; border-collapse: collapse; font-size: 13px; }
  .tbl th, .tbl td { text-align: left; padding: 8px 12px; border-bottom: 1px solid #1a2236; }
  .tbl th { color: #93a1b3; font-weight: 600; }
  .tbl td.num, .tbl th.num { text-align: right; font-variant-numeric: tabular-nums; }
  ul.news { list-style: none; margin: 0; padding: 4px 0; max-height: 460px; overflow: auto; }
  ul.news li { padding: 9px 16px; border-bottom: 1px solid #1a2236; font-size: 13px; display: flex; justify-content: space-between; gap: 10px; }
  ul.news a { color: #cdd9e5; text-decoration: none; }
  ul.news a:hover { color: #58a6ff; text-decoration: underline; }
  ul.news .src { color: #5b6b82; font-size: 11px; text-transform: uppercase; flex: none; }
  .muted { color: #5b6b82; }
  footer { color: #5b6b82; font-size: 11px; margin-top: 16px; }
</style></head>
<body><div class="wrap">
  <div class="top"><h1>⚽ FIFA World Cup 2026 — Live Dashboard</h1><span class="badge ${isSample ? "sample" : "live"}">${isSample ? "SAMPLE DATA" : "LIVE"}</span></div>
  <div class="sub">Polymarket implied odds × RSSHub headlines · prediction-market prices, not forecasts</div>
  <div class="kpis">${kpiCards}</div>
  <div class="grid">
    <div class="card"><h2>Markets — implied probability &amp; volume (Perspective)</h2><div id="pv">Loading Perspective…</div></div>
    <div class="card"><h2>Latest headlines</h2><ul class="news">${newsItems}</ul></div>
  </div>
  <footer>Sources: gamma-api.polymarket.com · rsshub.app (Google-News fallback). Re-run a source cell to refresh. Rendered by the world-cup-2026 Spur App.</footer>
</div>${scriptTag}</body></html>`;

await Deno.jupyter.display({ "text/html": html }, { raw: true });

<!doctype html> 
 
 
 
 ⚽ FIFA World Cup 2026 — Live Dashboard LIVE 
 Polymarket implied odds × RSSHub headlines · prediction-market prices, not forecasts 
 Market favorite Will 4+ matches go to extra time during the 2026 FIFA World Cup? 95.1% Markets tracked 11 Total volume $116,408 Headlines 30 
 
 Markets — implied probability & volume (Perspective) Loading Perspective… 
 Latest headlines World Cup 2026 live updates: Colombia returns against debutant Uzbekistan; Ghana beats Panama on late goal, England downs Croatia, Portugal and DR Congo draw in earlier games - NBC News rss Argentina vs Algeria Extended Highlights | 2026 FIFA World Cup™ - FOX Sports rss Ghana beat Panama with stoppage-time winner at World Cup 2026: Live updates and reaction - The New York Times rss FIFA World Cup 2026 MD7 recap: Ronaldo's Portugal frustrated by Congo DR, England hit 4 past Croatia, Panama hold Ghana - ESPN rss Uzbekistan v Colombia: World Cup 2026 – live - The Guardian rss Uzbekistan vs Colombia live updates: World Cup latest as Munoz gives Colombia first-half lead - The New York Times rss Uzbekistan vs. Colombia LIVE: World Cup 2026 updates as James Rodriguez and co. look for winning start - ESPN rss Re-ranking the 48 World Cup teams after day six: France and Argentina justify their top spots - The Athletic - The New York Times rss World Cup 2026: What you need to know about all 48 teams - ESPN rss A new No 1! Re-ranking the 48 World Cup teams after day five of the tournament - The Athletic - The New York Times rss What is happening with World Cup ticket prices? - BBC rss World Cup 2026: guide to all 1,248 players - The Guardian rss A team-by-team guide to the 2026 World Cup: What to expect and who to watch - The Athletic - The New York Times rss 'Scared to take him off' - Ronaldo struggles after fellow superstars sparkle - BBC rss World Cup fans from around the globe share 1st US food experiences on social media - ABC News - Breaking News, Latest News and Videos rss <a href="https://news.google.com/rss/articles/CBMiowFBVV95cUxOaThkUlc3R2kzcG1GYjJxUzVGb1BQcVQyaWUwT1hHSGVjNlNRZ3dqNmtYMUxYNVl3Q1AwOFQ3Tnp1cGU3WmloZHU0ZTdKUTRlWjVnMXJadzdCSmZpeGp5dHdJcXZMcHlLcEpDYkc2TkJLRldzUFFVV1NBeFdzbmFYTHZWYkJHdTdxVzJBVmY4WFJyOHdRWVlqWjlWR0JKd1hqaUU00gGoAUFVX3lxTE95M1pfaWh5WGQyc0w2VDFoTmQ2VWplR21Uc3hSZ3hsT2J5cXA0TDc0SnZjOGozLXhxRVJNU1VpbW9YUFd4eFZJbTZ3elp6WUpWWFoyQVUtMGsxX1BTUjdIYmxpWmREWjZyTWtQbExOTXB3cTI0NV91LXprRHJORXZqTnZQUVM4eVNxc1B1WnR2Qk91WU1qbW5KNThlSGFaUmV5dGtPMThNcw?oc=5" target="_blank" rel="noopener">Ghana beat Panama 1–0 in chaotic, charged World Cup Group L match - Al Jazeera rss World Cup 2026 schedule: Here's where to watch every match for free - Yahoo Sports rss Portugal held to 1-1 draw by DR Congo in their World Cup opener - Reuters rss 
 
 Sources: gamma-api.polymarket.com · rsshub.app (Google-News fallback). Re-run a source cell to refresh. Rendered by the world-cup-2026 Spur App.

In [ ]:
SELECT * FROM "graph-index.pointer".main LIMIT 100;


CatalogException: Catalog Error: Table with name "graph-index.pointer.main" does not exist because schema "graph-index.pointer" does not exist.

In [ ]:
SELECT * FROM session_metadata.main LIMIT 100;


In [ ]:
SELECT * FROM session_metadata.main LIMIT 100;


In [2]:
ATTACH IF NOT EXISTS '/Volumes/Projects/spur/.spur/analyst.duckdb' AS analyst (READ_ONLY);
SELECT * FROM analyst.main.__duckpgq_internal LIMIT 100;


property_graph,table_name,label,is_vertex_table,source_table,source_pk,source_fk,destination_table,destination_pk,destination_fk,discriminator,sub_labels,catalog,schema,source_catalog,source_schema,destination_catalog,destination_schema,properties,column_aliases
code,duckpgq_external_nodes,external,True,NaN,<NA>,<NA>,NaN,<NA>,<NA>,None,<NA>,,,NaN,NaN,NaN,NaN,"[stable_symbol_id, node_id, qualified_name, entity_name, symbol_kind, file_path]",[]
code,duckpgq_nodes,duckpgq_nodes,True,NaN,<NA>,<NA>,NaN,<NA>,<NA>,None,<NA>,,,NaN,NaN,NaN,NaN,"[stable_symbol_id, node_id, qualified_name, entity_name, symbol_kind, file_path]",[]
code,duckpgq_cross_crate_call_edges,cross_crate_calls,False,duckpgq_nodes,[stable_symbol_id],[source_stable_id],duckpgq_nodes,[stable_symbol_id],[target_stable_id],None,<NA>,,,,,,,"[edge_kind, relation, confidence, bind_method]",[]
code,duckpgq_import_edges,imports,False,duckpgq_nodes,[stable_symbol_id],[source_stable_id],duckpgq_external_nodes,[stable_symbol_id],[target_stable_id],None,<NA>,,,,,,,"[edge_kind, relation, confidence, bind_method]",[]
code,duckpgq_edges,duckpgq_edges,False,duckpgq_nodes,[stable_symbol_id],[source_stable_id],duckpgq_nodes,[stable_symbol_id],[target_stable_id],None,<NA>,,,,,,,"[edge_kind, relation, confidence, bind_method]",[]


In [3]:
ATTACH IF NOT EXISTS '/Volumes/Projects/spur/.spur/analyst.duckdb' AS analyst (READ_ONLY);
SELECT * FROM analyst.main.duckpgq_nodes LIMIT 100;


stable_symbol_id,node_id,qualified_name,entity_name,symbol_kind,file_path
fcb67af75e1f7ae9,54905,cell://2762e6e6-65c9-4d6e-865d-fb53448d82ef::4. Piece 3 — Predicate sub-axes ⚠️,4. Piece 3 — Predicate sub-axes ⚠️,section,docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-design.ipynb
02631e05b9b52fb3,550,cell://6bfdfe64-3186-4def-a206-48503fda7be9,cell://6bfdfe64-3186-4def-a206-48503fda7be9,cell,docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-spec-live-evidence.ipynb
16685b0bd0ad0084,4891,cell://6bfdfe64-3186-4def-a206-48503fda7be9::sx,sx,constant,docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-spec-live-evidence.ipynb
17c72ab569f2e1ad,5175,cell://6bfdfe64-3186-4def-a206-48503fda7be9::pts,pts,constant,docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-spec-live-evidence.ipynb
217f913ea6262099,7300,cell://8ed1c3da-cb26-4c08-8ae1-a7d195814e7a,cell://8ed1c3da-cb26-4c08-8ae1-a7d195814e7a,cell,docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-spec-live-evidence.ipynb
42c124240efab8b4,14617,cell://6bfdfe64-3186-4def-a206-48503fda7be9::area,area,constant,docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-spec-live-evidence.ipynb
4cc95ba6cc1095f8,16781,cell://6bfdfe64-3186-4def-a206-48503fda7be9::X,X,constant,docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-spec-live-evidence.ipynb
657158fca55029d2,22029,cell://6bfdfe64-3186-4def-a206-48503fda7be9::conf,conf,constant,docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-spec-live-evidence.ipynb
7035816685063e5a,24380,cell://6bfdfe64-3186-4def-a206-48503fda7be9::sy,sy,constant,docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-spec-live-evidence.ipynb
797ba72d468d33a4,26464,cell://6bfdfe64-3186-4def-a206-48503fda7be9::html,html,constant,docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-spec-live-evidence.ipynb


In [4]:
SELECT * FROM read_json_auto('/Volumes/Projects/spur/.spur/commit-index.json') LIMIT 100;


In [11]:
SELECT * FROM read_json_auto('/Volumes/Projects/spur/.spur/commit-index.json') LIMIT 100;
